# **similarity by bert**

In [ ]:

import torch
import pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity


sentences = [
             "Three years later, the coffin was still full of Jello.",
             "The fish dreamed of escaping the fishbowl and into the toilet where he saw his friend go.",
             "The person box was packed with jelly many dozens of months later.",
             "Standing on one's head at job interviews forms a lasting impression.",
             "It took him a month to finish the meal.",
             "Finishing the meal took him 3 weeks.",
             "Five years later, the suitcase was still packed with confetti.",
             "Five years later, the suitcase was still packed with confetti.",
             "The bird wished to fly beyond the cage and into the sky where it saw its friend soar.",
             "The drawer was overflowing with paperclips many months down the line.",
             "Wearing mismatched socks to meetings always makes a statement.",
             "It took her two weeks to paint the portrait.",
             "Painting the portrait required her a fortnight."

]




*   nous téléchargeons et chargeons le tokenize
*   nous téléchargeons et chargeons l' encouder

In [ ]:

model_name = 'sentence-transformers/bert-base-nli-mean-tokens'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

La fonction compute_tokens prend une liste de phrases en entrée et les transforme en tokens utilisables par un modèle BERT.
Elle utilise le tokenizer pour convertir chaque phrase en input_ids  et attention_mask .

In [ ]:
def compute_tokens(sentences, tokenizer):

    input_ids = []
    attention_mask = []


    for sentence in sentences:
        sentence_encoding = tokenizer.encode_plus(
            sentence,
            max_length=128,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )

        input_ids.append(sentence_encoding['input_ids'][0])
        attention_mask.append(sentence_encoding['attention_mask'][0])

    input_ids = torch.stack(input_ids)
    attention_mask = torch.stack(attention_mask)


    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask
    }


La fonction compute_sentence_vector calcule une représentation vectorielle moyenne pour une ou plusieurs phrases.

In [ ]:
def compute_sentence_vector(tokens, model):

    last_hidden_state, pooled_output = model(**tokens, return_dict=False)

    attention_mask = tokens['attention_mask'].unsqueeze(-1).expand(last_hidden_state.shape).float()
    masked_embeddings = last_hidden_state * attention_mask


    summed = torch.sum(masked_embeddings, dim=1)
    counts = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
    mean_pooled_embedding = summed / counts

    return mean_pooled_embedding

La fonction compute_similarity calcule la similarité entre une phrase de référence (la première de la liste)et toutes les autres phrases en utilisant le modèle BERT pour générer des embeddings.

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def compute_similarity(sentences, tokenizer, model, threshold=0.8):
    sentences_tokens = compute_tokens(sentences, tokenizer)
    sentences_embeddings = compute_sentence_vector(sentences_tokens, model)

    sentences_embeddings_detached = sentences_embeddings.detach().numpy()
    similarity_scores = cosine_similarity([sentences_embeddings_detached[0]], sentences_embeddings_detached[1:])
    d = {
        'column-1': [sentences[0] for _ in range(len(sentences)-1)],
        'column-2': [sent for sent in sentences[1:]],
        'scores': similarity_scores[0]
    }

    output = pd.DataFrame(data=d)
    output['similarity_label'] = output['scores'].apply(lambda score: 'Similar' if score > 0.5 else 'Not Similar')

    return output



C'est terminé ! Nous affichons maintenant Le résultat est un DataFrame contenant chaque paire de phrases, leur score de similarité cosinus,et une étiquette indiquant si elles sont similaires ou non


In [ ]:
output = compute_similarity(sentences, tokenizer, model)
output

,column-1,column-2,scores,similarity_label
0,"Three years later, the coffin was still full o...",The fish dreamed of escaping the fishbowl and ...,0.330889,Not Similar
1,"Three years later, the coffin was still full o...",The person box was packed with jelly many doze...,0.721926,Similar
2,"Three years later, the coffin was still full o...",Standing on one's head at job interviews forms...,0.174755,Not Similar
3,"Three years later, the coffin was still full o...",It took him a month to finish the meal.,0.447097,Not Similar
4,"Three years later, the coffin was still full o...",Finishing the meal took him 3 weeks.,0.579585,Similar
5,"Three years later, the coffin was still full o...","Five years later, the suitcase was still packe...",0.699263,Similar
6,"Three years later, the coffin was still full o...","Five years later, the suitcase was still packe...",0.699263,Similar
7,"Three years later, the coffin was still full o...",The bird wished to fly beyond the cage and int...,0.114647,Not Similar
8,"Three years later, the coffin was still full o...",The drawer was overflowing with paperclips man...,0.596767,Similar
9,"Three years later, the coffin was still full o...",Wearing mismatched socks to meetings always ma...,0.347937,Not Similar
